In [0]:
# =============================================================
# Notebook : 03_ecom_silver.py
# Purpose  : Bronze → Silver for e-commerce clickstream
# Source   : bronze/stream/ecom_clickstream/
# Target   : silver/fact_ecom_events/ (Delta)
# =============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

BRONZE_PATH = "abfss://bronze@walmartdata.dfs.core.windows.net/stream/ecom_clickstream/*/*/"
SILVER_PATH = "abfss://silver@walmartdata.dfs.core.windows.net/fact_ecom_events/"

print("Reading Bronze clickstream...")
df_raw = spark.read.json(BRONZE_PATH)
print(f"Raw rows: {df_raw.count():,}")
df_raw.printSchema()

In [0]:
# ── Schema enforcement + enrichment ──────────────────────────
df_typed = (
    df_raw
    .select(
        F.col("event_id").cast(StringType()),
        F.col("session_id").cast(StringType()),
        F.col("customer_id").cast(StringType()),
        F.col("event_type").cast(StringType()),
        F.to_timestamp("timestamp").alias("event_ts"),
        F.col("platform").cast(StringType()),
        F.col("city").cast(StringType()),
        F.col("device_type").cast(StringType()),
        F.col("sku").cast(StringType()),
        F.col("product_name").cast(StringType()),
        F.col("category").cast(StringType()),
        F.col("price").cast(DoubleType()),
        F.col("search_query").cast(StringType()),
        F.col("order_value").cast(DoubleType()),
        F.to_timestamp("ingested_at").alias("ingested_at"),
        F.current_timestamp().alias("silver_processed_at"),
        F.lit("ecom_bronze_to_silver_v1").alias("pipeline_version"),
    )
    # ── Derived columns ───────────────────────────────────────
    .withColumn("event_date",  F.to_date("event_ts"))
    .withColumn("event_hour",  F.hour("event_ts"))
    .withColumn("event_month", F.month("event_ts"))
    .withColumn("event_year",  F.year("event_ts"))
    .withColumn("is_purchase",
                F.col("event_type") == "PURCHASE")
    .withColumn("is_add_to_cart",
                F.col("event_type") == "ADD_TO_CART")
    .withColumn("is_search",
                F.col("event_type") == "SEARCH")
    .withColumn("platform_type",
                F.when(F.col("platform").contains("app"), "App")
                 .otherwise("Web"))
)

print(f"Typed rows: {df_typed.count():,}")

In [0]:
# ── Data quality checks ───────────────────────────────────────
total = df_typed.count()
print("=" * 50)
print("  DATA QUALITY — Ecom Clickstream")
print("=" * 50)
print(f"  Total events    : {total:,}")
print(f"  Null event_id   : {df_typed.filter(F.col('event_id').isNull()).count():,}")
print(f"  Null session_id : {df_typed.filter(F.col('session_id').isNull()).count():,}")
print(f"  Null event_ts   : {df_typed.filter(F.col('event_ts').isNull()).count():,}")
print("=" * 50)

print("\nEvent type distribution:")
df_typed.groupBy("event_type") \
        .count() \
        .orderBy("count", ascending=False) \
        .display()

In [0]:
# ── Deduplicate on event_id ───────────────────────────────────
df_deduped = (
    df_typed
    .withColumn(
        "row_num",
        F.row_number().over(
            Window.partitionBy("event_id")
                  .orderBy(F.col("ingested_at").desc())
        )
    )
    .filter(F.col("row_num") == 1)
    .drop("row_num")
    .filter(F.col("event_id").isNotNull())
    .filter(F.col("session_id").isNotNull())
    .filter(F.col("event_ts").isNotNull())
)

print(f"After dedup: {df_deduped.count():,} rows")

In [0]:
# ── Write Silver Delta ────────────────────────────────────────
(
    df_deduped
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("event_year", "event_month", "platform_type")
    .save(SILVER_PATH)
)

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS walmart_silver.fact_ecom_events
    USING DELTA
    LOCATION '{SILVER_PATH}'
""")

print(f"✅ walmart_silver.fact_ecom_events registered")
print(f"   Rows: {spark.table('walmart_silver.fact_ecom_events').count():,}")

# Funnel analysis
print("\nConversion funnel:")
spark.sql("""
    SELECT
        event_type,
        COUNT(DISTINCT session_id)  AS unique_sessions,
        COUNT(*)                    AS total_events
    FROM walmart_silver.fact_ecom_events
    GROUP BY event_type
    ORDER BY total_events DESC
""").display()